In [1]:
# Stream IMERG data from Kerchunk virtual data stores
#
# 8/27/2026 JRS
# Use nasa-gesdisc-kerchunk environment

import warnings
import earthaccess
import xarray as xr

auth = earthaccess.login(strategy="netrc")
bearer_token = auth.token["access_token"]

In [2]:
# Create virtual dataset loader function (from Chris B's How To)
def get_vds(parq: str, chunks: dict={}, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=xr.SerializationWarning)
        return xr.open_dataset(
            "reference://",
            engine="zarr",
            chunks=chunks,
            backend_kwargs={
                "storage_options": {
                    "fo": str(parq),
                    "remote_protocol": "https",
                    "asynchronous": True,
                    "remote_options": {
                        "headers": {"Authorization": f'Bearer {bearer_token}'},
                        "asynchronous": True,
                    }
                },
                "consolidated": True
            },
            **kwargs
        )

In [3]:
# Read in kerchunk files of IMERG data (one year each) and concatenate them
vds2010 = get_vds("https://data.gesdisc.earthdata.nasa.gov/browse/kerchunk/GPM_L3/GPM_3IMERGDF.07/2010.parq")
vds2011 = get_vds("https://data.gesdisc.earthdata.nasa.gov/browse/kerchunk/GPM_L3/GPM_3IMERGDF.07/2011.parq")

vds_concat = xr.concat([vds2010, vds2011], dim="time")
vds_concat.attrs["BeginDate"] = str(vds_concat.time.min().dt.date.values)
vds_concat.attrs["EndDate"] = str(vds_concat.time.max().dt.date.values)
print(vds_concat)


<xarray.Dataset> Size: 170GB
Dimensions:                         (time: 730, lon: 3600, lat: 1800, nv: 2)
Coordinates:
  * time                            (time) datetime64[ns] 6kB 2010-01-01 ... ...
  * lon                             (lon) float32 14kB -179.9 -179.9 ... 179.9
  * lat                             (lat) float64 14kB -89.95 -89.85 ... 89.95
Dimensions without coordinates: nv
Data variables:
    MWprecipitation                 (time, lon, lat) float32 19GB dask.array<chunksize=(1, 3600, 900), meta=np.ndarray>
    MWprecipitation_cnt             (time, lon, lat) float32 19GB dask.array<chunksize=(1, 3600, 900), meta=np.ndarray>
    MWprecipitation_cnt_cond        (time, lon, lat) float32 19GB dask.array<chunksize=(1, 3600, 900), meta=np.ndarray>
    precipitation                   (time, lon, lat) float32 19GB dask.array<chunksize=(1, 3600, 900), meta=np.ndarray>
    precipitation_cnt               (time, lon, lat) float32 19GB dask.array<chunksize=(1, 3600, 900), meta=np.